RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [8]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [11]:
###
def process_all_pdfs(pdf_directory):
    """Process all pdfs directly"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    # Find all pdfs
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\n Processing: {pdf_file.name} ")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata['file_type'] = "pdf"
            
            all_documents.extend(documents)
        
        except Exception as exception:
            print(f"Error {str(exception)}")
        
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_documents = process_all_pdfs("../data")

Found 3 PDF files to process

 Processing: attention_mechanisms_basic_information.pdf 

 Processing: embeddings_basic_information.pdf 

 Processing: llm_basic_information.pdf 
Total documents loaded: 3


In [13]:
all_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-11T07:13:12+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-11T07:13:12+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf_files/attention_mechanisms_basic_information.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'attention_mechanisms_basic_information.pdf', 'file_type': 'pdf'}, page_content='Basic Information About Attention Mechanisms\nAttention mechanisms are techniques used in machine learning models to decide which parts\nof the input are most important for a given task. Instead of treating every word or token\nequally, attention allows the model to focus more strongly on relevant information.\nIn language models, attention helps connect words across a sentence or document. For\nexample, it can help a model understand which noun a pronoun refers to, or which ear

In [14]:
###Textplitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap = 200):
    """Split documents into smaller chunks for better RAG Performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n","\n"," ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\n Example chunk:")
        print(f"Content {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

chunks = split_documents(all_documents)

Split 3 documents into 3 chunks

 Example chunk:
Content Basic Information About Attention Mechanisms
Attention mechanisms are techniques used in machine learning models to decide which parts
of the input are most important for a given task. Instead of trea
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-11T07:13:12+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-11T07:13:12+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf_files/attention_mechanisms_basic_information.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'attention_mechanisms_basic_information.pdf', 'file_type': 'pdf'}
